# Y2 · Arabic NLI training pairs (Youssef) · GPU runtime
**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all** (about 45-60 min, keep the tab open).
At the end two files download: `ar_train.jsonl` (final, for Rathi) and `ar_train_translated.jsonl`. Put them in `data/nli/` in your repo and push.

In [ ]:
!git clone -q https://github.com/himaallu/Group-Project.git /content/Group-Project
%cd /content/Group-Project
!pip install -q -U transformers accelerate bitsandbytes
!mkdir -p src/translate data/nli

Scripts (same as in the repo; written here so the notebook works even before your PR is merged)

In [ ]:
%%writefile /content/Group-Project/src/nli/build_ar_pairs.py
"""Build Arabic NLI data from cleaned X-FACT Arabic (task Y2, part 1).

What it does
    1. Gold pool: picks the claims for hand-labelled Arabic gold pairs. Arabic val
       has only 93 claims with snippets, so 57 train claims are added (10 supported,
       7 unverified, 40 contradicted, seed 42) to reach 150. These claims are then
       kept OUT of the training pairs, so gold and train never share a claim.
    2. Weak training pairs from the remaining train claims (same rules as R4):
         supported claim    + its relevant snippet  -> entail
         contradicted claim + its relevant snippet  -> contradict
         any claim          + snippets of 2 random other claims -> neutral
       "Relevant" = the claim's snippets with the most word overlap (at most 2 per
       claim, overlap >= 0.2), since there are no Arabic reranker scores yet.
       Unverified claims give no entail/contradict pairs (their snippets do not
       settle the claim). Contradict and neutral are capped so classes stay balanced;
       entail is topped up later with translated English pairs (translate_pairs.py).

Inputs
    data/processed/ar/xfact_ar_{train,val}.jsonl   (cleaned with src/data/clean.py)

Outputs
    data/nli/ar_gold_pool.jsonl     shared-format claims for src.nli.sample_gold
    data/nli/ar_train_xfact.jsonl   weak pairs, same fields as data/nli/gold_val.jsonl
                                    (id, claim_id, snippet_idx, language, claim, snippet,
                                    label) plus "origin": "xfact_weak"

How to run (repo root)
    python -m src.nli.build_ar_pairs --limit 10          # 10-claim test run
    python -m src.nli.build_ar_pairs
    python -m src.nli.sample_gold data/nli/ar_gold_pool.jsonl --n 150 \
        --out results/Y2/gold_sheet_ar.csv
"""

from __future__ import annotations

import argparse
import random
import re
from collections import Counter

from src.data.common import SEED, read_jsonl, write_jsonl

WORD = re.compile(r"\w+", re.UNICODE)
EXTRA_TRAIN_FOR_GOLD = {"supported": 10, "unverified": 7, "contradicted": 40}
MAX_SNIPPETS_PER_CLAIM = 2
MIN_OVERLAP = 0.2
NEUTRAL_PER_CLAIM = 2
WEAK_LABEL = {"supported": "entail", "contradicted": "contradict"}


def overlap(claim: str, snippet: str) -> float:
    """Share of the claim's distinct words that also appear in the snippet."""
    cw = set(WORD.findall(claim.lower()))
    return len(cw & set(WORD.findall(snippet.lower()))) / len(cw) if cw else 0.0


def pick_gold_pool(train: list[dict], val: list[dict], rng: random.Random) -> list[dict]:
    """Return val claims plus a fixed, label-stratified sample of train claims.

    Args:
        train, val: cleaned shared-format records (claims with snippets only).
        rng: seeded random generator.
    Returns:
        list of records for the gold pool (val first, then the extra train claims).
    """
    extra = []
    for label, n in EXTRA_TRAIN_FOR_GOLD.items():
        pool = sorted((r for r in train if r["label"] == label), key=lambda r: r["id"])
        extra += rng.sample(pool, min(n, len(pool)))
    return val + extra


def pair(rec: dict, idx: int, label: str, snippet_rec: dict | None = None) -> dict:
    """Make one NLI pair in the gold_val.jsonl format.

    Args:
        rec: the claim's record.
        idx: snippet index inside the record the snippet comes from.
        label: entail / contradict / neutral.
        snippet_rec: record that owns the snippet (another claim for neutral pairs).
    """
    src = snippet_rec or rec
    tag = "s" if snippet_rec is None else f"r_{src['id']}_"
    return {"id": f"{rec['id']}_{tag}{idx}", "claim_id": rec["id"], "snippet_idx": idx,
            "language": "ar", "claim": rec["claim"], "snippet": src["evidence"][idx]["text"],
            "label": label, "origin": "xfact_weak"}


def relevant_snippets(rec: dict) -> list[int]:
    """Indices of the claim's most relevant snippets (by word overlap, at most 2)."""
    scored = sorted(((overlap(rec["claim"], s["text"]), i) for i, s in enumerate(rec["evidence"])),
                    reverse=True)
    return [i for score, i in scored[:MAX_SNIPPETS_PER_CLAIM] if score >= MIN_OVERLAP]


def build_weak_pairs(train: list[dict], rng: random.Random, max_per_class: int) -> list[dict]:
    """Make entail / contradict / neutral weak pairs from train claims.

    Contradict and neutral are capped at max_per_class (random, seeded); entail
    keeps everything (it is the rare class).
    """
    by_label = {"entail": [], "contradict": [], "neutral": []}
    for rec in train:
        if rec["label"] in WEAK_LABEL:
            for i in relevant_snippets(rec):
                by_label[WEAK_LABEL[rec["label"]]].append(pair(rec, i, WEAK_LABEL[rec["label"]]))
        # neutral: snippets from NEUTRAL_PER_CLAIM different, random claims
        for _ in range(NEUTRAL_PER_CLAIM):
            other = rng.choice(train)
            while other["id"] == rec["id"]:
                other = rng.choice(train)
            by_label["neutral"].append(pair(rec, rng.randrange(len(other["evidence"])), "neutral", other))
    for lab in ("contradict", "neutral"):
        rng.shuffle(by_label[lab])
        by_label[lab] = by_label[lab][:max_per_class]
    pairs = by_label["entail"] + by_label["contradict"] + by_label["neutral"]
    rng.shuffle(pairs)
    return pairs


def main() -> None:
    """Write the gold pool and the weak X-FACT pairs, and print counts."""
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--train", default="data/processed/ar/xfact_ar_train.jsonl")
    ap.add_argument("--val", default="data/processed/ar/xfact_ar_val.jsonl")
    ap.add_argument("--pool_out", default="data/nli/ar_gold_pool.jsonl")
    ap.add_argument("--pairs_out", default="data/nli/ar_train_xfact.jsonl")
    ap.add_argument("--max_per_class", type=int, default=1000)
    ap.add_argument("--limit", type=int, default=None, help="use only N train claims (test run)")
    args = ap.parse_args()

    rng = random.Random(SEED)
    train = [r for r in read_jsonl(args.train) if r.get("evidence")]
    val = [r for r in read_jsonl(args.val) if r.get("evidence")]
    pool = pick_gold_pool(train, val, rng)
    pool_ids = {r["id"] for r in pool}
    train_rest = [r for r in train if r["id"] not in pool_ids][: args.limit]

    pairs = build_weak_pairs(train_rest, rng, args.max_per_class)
    if not args.limit:
        write_jsonl(pool, args.pool_out)
    write_jsonl(pairs, args.pairs_out if not args.limit else args.pairs_out + ".sample10")
    print(f"gold pool: {len(pool)} claims {dict(Counter(r['label'] for r in pool))}")
    print(f"train claims used: {len(train_rest)} (gold-pool claims excluded)")
    print(f"weak pairs: {len(pairs)} {dict(Counter(p['label'] for p in pairs))}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/Group-Project/src/translate/translate_pairs.py
"""Top up the Arabic NLI training data with translated English entail pairs (task Y2, part 2).

What it does
    X-FACT Arabic has very few true claims, so it gives only ~46 entail pairs.
    This script:
      1. picks English entail pairs from MultiFC train: a supported claim + its snippet
         with the most word overlap (overlap >= 0.5), seed 42;
      2. translates claim and snippet to Modern Standard Arabic with Qwen3-8B
         (4-bit, greedy, thinking off). The prompt keeps names, numbers and dates
         unchanged in meaning; an optional glossary (CSV en,ar) fixes how names are
         written, so the same person is spelled the same way everywhere;
      3. --merge: joins the X-FACT weak pairs and the translated pairs into the final
         data/nli/ar_train.jsonl, keeping the three classes balanced.
    Translation writes one line per finished pair, so a re-run after a Colab
    disconnect continues where it stopped.

Inputs
    data/processed/en/multifc_train.jsonl, data/nli/ar_train_xfact.jsonl,
    optional data/translate/glossary_en_ar.csv

Outputs
    data/nli/ar_train_translated.jsonl   translated pairs (+ "claim_en", "snippet_en")
    data/nli/ar_train.jsonl              final Arabic training pairs for Rathi's R4,
                                         same fields as data/nli/gold_val.jsonl

How to run (repo root, GPU runtime)
    python -m src.translate.translate_pairs --n 10            # test on 10 pairs
    python -m src.translate.translate_pairs --n 954           # full (~30-60 min on a T4)
    python -m src.translate.translate_pairs --merge
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import random
import re
from collections import Counter

from src.data.common import SEED, read_jsonl, write_jsonl

MODEL = "Qwen/Qwen3-8B"
WORD = re.compile(r"\w+", re.UNICODE)
MIN_OVERLAP = 0.5
MAX_SNIPPET_WORDS = 120  # long snippets are cut so the translation stays fast

PROMPT = (
    "Translate the following English text into Modern Standard Arabic.\n"
    "Rules: keep the exact meaning; keep every name, number and date (write numbers "
    "as digits); do not add, explain or drop anything; output only the translation.\n"
    "{glossary}\nText:\n{text}"
)


def overlap(claim: str, snippet: str) -> float:
    """Share of the claim's distinct words that also appear in the snippet."""
    cw = set(WORD.findall(claim.lower()))
    return len(cw & set(WORD.findall(snippet.lower()))) / len(cw) if cw else 0.0


def select_entail_pairs(records: list[dict], n: int, seed: int = SEED) -> list[dict]:
    """Pick n English entail pairs (supported claim + best-overlap snippet).

    Args:
        records: MultiFC train records in the shared format.
        n: number of pairs wanted.
    Returns:
        list of pair dicts in the gold_val.jsonl format (English).
    """
    pairs = []
    for rec in records:
        if rec["label"] != "supported" or not rec.get("evidence"):
            continue
        scores = [(overlap(rec["claim"], s["text"]), i) for i, s in enumerate(rec["evidence"])]
        best, idx = max(scores)
        if best >= MIN_OVERLAP:
            snippet = " ".join(rec["evidence"][idx]["text"].split()[:MAX_SNIPPET_WORDS])
            pairs.append({"id": f"{rec['id']}_s{idx}_tr", "claim_id": rec["id"], "snippet_idx": idx,
                          "language": "en", "claim": rec["claim"], "snippet": snippet,
                          "label": "entail"})
    random.Random(seed).shuffle(pairs)
    return pairs[:n]


def load_glossary(path: str | None) -> str:
    """Return glossary lines for the prompt ('English -> Arabic'), or '' if no file."""
    if not path or not os.path.exists(path):
        return ""
    with open(path, encoding="utf-8") as f:
        rows = list(csv.reader(f))
    lines = [f"{en} -> {ar}" for en, ar in rows[1:] if en and ar]
    return "Use these spellings:\n" + "\n".join(lines) + "\n" if lines else ""


def load_model():
    """Load Qwen3-8B in 4-bit (fits a free T4) with left padding for batched generation."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))
    return tok, model


def translate_batch(texts: list[str], tok, model, glossary: str, max_new_tokens: int = 320) -> list[str]:
    """Translate a batch of English texts to Arabic (greedy decoding, thinking disabled)."""
    import torch

    prompts = [tok.apply_chat_template([{"role": "user", "content": PROMPT.format(glossary=glossary, text=t)}],
                                       tokenize=False, add_generation_prompt=True, enable_thinking=False)
               for t in texts]
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    new = out[:, enc["input_ids"].shape[1]:]
    return [re.sub(r"<think>.*?</think>", "", t, flags=re.S).strip()
            for t in tok.batch_decode(new, skip_special_tokens=True)]


def run_translation(args) -> None:
    """Translate the selected pairs, appending to the output file (resumable)."""
    pairs = select_entail_pairs(read_jsonl(args.multifc), args.n)
    done = set()
    if os.path.exists(args.out):
        done = {json.loads(l)["id"] for l in open(args.out, encoding="utf-8") if l.strip()}
    todo = [p for p in pairs if p["id"] not in done]
    print(f"selected {len(pairs)} pairs, {len(done)} already done, {len(todo)} to translate")
    if not todo:
        return
    tok, model = load_model()
    glossary = load_glossary(args.glossary)
    os.makedirs(os.path.dirname(args.out), exist_ok=True)
    with open(args.out, "a", encoding="utf-8") as f:
        for start in range(0, len(todo), args.batch):
            batch = todo[start:start + args.batch]
            texts = [p["claim"] for p in batch] + [p["snippet"] for p in batch]
            ar = translate_batch(texts, tok, model, glossary)
            for p, claim_ar, snip_ar in zip(batch, ar[:len(batch)], ar[len(batch):]):
                f.write(json.dumps({**p, "language": "ar", "claim": claim_ar, "snippet": snip_ar,
                                    "claim_en": p["claim"], "snippet_en": p["snippet"],
                                    "origin": "multifc_translated"}, ensure_ascii=False) + "\n")
            f.flush()
            print(f"{min(start + args.batch, len(todo))}/{len(todo)} translated")


def merge(args) -> None:
    """Join X-FACT weak pairs and translated pairs into data/nli/ar_train.jsonl (balanced)."""
    rng = random.Random(SEED)
    pairs = read_jsonl(args.xfact_pairs) + read_jsonl(args.out)
    by_label = {lab: [p for p in pairs if p["label"] == lab] for lab in ("entail", "contradict", "neutral")}
    cap = max(len(by_label["entail"]), 1)  # entail is the smallest class after the top-up
    final = []
    for lab, items in by_label.items():
        rng.shuffle(items)
        final += items[:cap]
    rng.shuffle(final)
    write_jsonl(final, args.final)
    print(f"{args.final}: {len(final)} pairs {dict(Counter(p['label'] for p in final))} "
          f"origins {dict(Counter(p.get('origin') for p in final))}")


def main() -> None:
    """Command-line entry point: translate (default) or --merge."""
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--multifc", default="data/processed/en/multifc_train.jsonl")
    ap.add_argument("--xfact_pairs", default="data/nli/ar_train_xfact.jsonl")
    ap.add_argument("--out", default="data/nli/ar_train_translated.jsonl")
    ap.add_argument("--final", default="data/nli/ar_train.jsonl")
    ap.add_argument("--glossary", default="data/translate/glossary_en_ar.csv")
    ap.add_argument("--n", type=int, default=954)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--merge", action="store_true")
    args = ap.parse_args()
    merge(args) if args.merge else run_translation(args)


if __name__ == "__main__":
    main()

Arabic X-FACT data: from main if your Y1 PR is merged, otherwise upload `Y1_finish.zip` when asked

In [ ]:
import os
if not os.path.exists('data/processed/ar/xfact_ar_train.jsonl'):
    from google.colab import files
    print('Upload Y1_finish.zip'); up = files.upload()
    !unzip -oq Y1_finish.zip -d .
!python -m src.nli.build_ar_pairs

Translate: 10 pairs first, look at them, then the rest

In [ ]:
!python -m src.translate.translate_pairs --n 10
import json
for l in list(open('data/nli/ar_train_translated.jsonl', encoding='utf-8'))[:3]:
    r = json.loads(l); print(r['claim_en'], '\n ->', r['claim'], '\n')

In [ ]:
!python -m src.translate.translate_pairs --n 954 --batch 8

In [ ]:
!python -m src.translate.translate_pairs --merge
from google.colab import files
files.download('data/nli/ar_train.jsonl')
files.download('data/nli/ar_train_translated.jsonl')